In [ ]:
pip install -qU langchain-pinecone pinecone-client

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -q langchain-community pypdf

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


## Load pdf

In [14]:
from langchain_community.document_loaders import PyPDFLoader

# 1. Point to your PDF file
loader = PyPDFLoader("ocpp-1.6 edition 2.pdf")

# 2. Extract the text and page numbers
pages = loader.load()

# 3. Check your work
print(f"✅ Successfully extracted {len(pages)} pages.")
print(f"Example text from Page 1: {pages[0].page_content[:200]}...")

✅ Successfully extracted 116 pages.
Example text from Page 1: Open Charge Point Protocol 1.6
edition 2 FINAL, 2017-09-28...


### Splited the data into text chunk

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Max characters per chunk
    chunk_overlap=100,  # "Context bridge" between chunks
    length_function=len,
    is_separator_regex=False,
)

# 2. Split the documents we loaded from the PDF
# 'pages' is the variable from your PyPDFLoader step
chunks = text_splitter.split_documents(pages)

print(f"✅ Your {len(pages)} pages have been turned into {len(chunks)} smaller chunks.")

✅ Your 116 pages have been turned into 297 smaller chunks.


In [18]:
chunks[2]

Document(metadata={'producer': 'edition 2 FINAL, 2017-09-28', 'creator': 'Asciidoctor PDF 1.5.0.alpha.15, based on Prawn 2.2.2', 'creationdate': '2017-09-28T17:13:02+02:00', 'title': 'Open Charge Point Protocol 1.6', 'author': 'edition 2 FINAL, 2017-09-28', 'moddate': '2017-09-28T17:13:02+02:00', 'source': 'ocpp-1.6 edition 2.pdf', 'total_pages': 116, 'page': 1, 'page_label': 'ii'}, page_content='2.4. References. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \xa07\n3. Introduction . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \xa08\n3.1. Edition 2. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . \xa08\n3.2. Document structure. . . . .

In [42]:
import os
import getpass

# This creates a secure input box in your notebook
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("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")

print("Environment Variable set!")
print(f"API Key present: {bool(os.environ.get('NVIDIA_API_KEY'))}")

Environment Variable set!
API Key present: False


In [47]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

# 1. Initialize the embeddings model
# This tells LangChain to use NVIDIA's API for the math
nvidia_embeddings = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5", 
                                     nvidia_api_key="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",
                                     truncate="END")


In [48]:
try:
    test_vector = nvidia_embeddings.embed_query("hello")
    print(f"🚀 It worked! Vector length: {len(test_vector)}")
except Exception as e:
    print(f"⚠️ Still failing. Error: {e}")

🚀 It worked! Vector length: 1024


In [53]:
result= nvidia_embeddings.embed_query("hello how are!")

In [54]:
len(result)

1024

#### Similarity search

In [57]:
v1 = nvidia_embeddings.embed_query("King")
v2 = nvidia_embeddings.embed_query("Queen")
v3 = nvidia_embeddings.embed_query("Apple")

# The lengths are all 1024
print(len(v1), len(v2), len(v3))

# But the math shows King and Queen are "closer" than King and Apple
from numpy import dot
from numpy.linalg import norm

def similarity(a, b):
    return dot(a, b)/(norm(a)*norm(b))

print(f"King vs Queen Similarity: {similarity(v1, v2):.4f}")
print(f"King vs Apple Similarity: {similarity(v1, v3):.4f}")

1024 1024 1024
King vs Queen Similarity: 0.7188
King vs Apple Similarity: 0.5944


#### PIncone

In [61]:
# 2. Set the PINECONE Key (The Memory)
if not os.environ.get("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("pcsk_5RHKmd_7r2wcyCjHsdWiH7LR3omkwKoehxx6xxJZxGhboxesVjHYcAgaxGqNphXK5mo8CR")

# Verification Check
print(f"✅ NVIDIA Key set: {bool(os.environ.get('NVIDIA_API_KEY'))}")
print(f"✅ Pinecone Key set: {bool(os.environ.get('PINECONE_API_KEY'))}")

✅ NVIDIA Key set: False
✅ Pinecone Key set: False


In [ ]:
%pip install pinecone

In [65]:
from pinecone import Pinecone

# Use the exact name you chose on the Pinecone website
INDEX_NAME = "vdb-rag-cloud" 

# Initialize the Pinecone client
pc = Pinecone(api_key="pcsk_5RHKmd_7r2wcyCjHsdWiH7LR3omkwKoehxx6xxJZxGhboxesVjHYcAgaxGqNphXK5mo8CR")

# # Connect to your specific index
index = pc.Index(INDEX_NAME)

# # Check the cloud status
print("--- Pinecone Status ---")
print(index.describe_index_stats())

--- Pinecone Status ---
{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


##### PineconeVectorStore

In [ ]:
from langchain_pinecone import PineconeVectorStore

test_store = PineconeVectorStore(index_name=INDEX_NAME,
                                embedding=nvidia_embeddings,
                                pinecone_api_key="pcsk_5RHKmd_7r2wcyCjHsdWiH7LR3omkwKoehxx6xxJZxGhboxesVjHYcAgaxGqNphXK5mo8CR")
print("🔑 Key Authentication Verified!")

🔑 Key Authentication Verified!


##### crete embeding from chunk docm

In [83]:
from langchain_pinecone import PineconeVectorStore
import os

os.environ["PINECONE_API_KEY"] = "pcsk_5RHKmd_7r2wcyCjHsdWiH7LR3omkwKoehxx6xxJZxGhboxesVjHYcAgaxGqNphXK5mo8CR"
# 1. Start the upload process
print(f"📤 Processing {len(chunks)} chunks...")

vector_store = PineconeVectorStore.from_documents(
    
    documents=chunks,           # Your 297 text chunks
    embedding=nvidia_embeddings,       # Your NVIDIA embedding object (1024 dims)
    index_name=INDEX_NAME,
    namespace="occp-v1.6"
)

📤 Processing 297 chunks...


In [84]:
# Check your Pinecone Index directly
stats = index.describe_index_stats()

print("--- CLOUD STORAGE VERIFIED ---")
print(f"Total Vectors stored: {stats['total_vector_count']}")
print(f"Namespace used: {list(stats['namespaces'].keys())[0]}")

--- CLOUD STORAGE VERIFIED ---
Total Vectors stored: 594
Namespace used: occp-v1.6


#### if you have already an index, you can load like this 

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 1. Connect to the index you already populated
vector_store = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=nvidia_embeddings,
    namespace="occp-v1.6", # Make sure this matches the name you used during upload
)

print("🔗 Connection re-established! The 'Brain' and 'Memory' are linked.")

🔗 Connection re-established! The 'Brain' and 'Memory' are linked.


In [87]:
query = "Who is the mother of dragons?"

# k=3 tells it to bring back the top 3 most similar paragraphs
docs = vector_store.similarity_search(query, k=3)

print(f"🔍 Found {len(docs)} relevant chunks.")
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} (Page {doc.metadata.get('page', 'N/A')}) ---")
    print(doc.page_content[:250] + "...")

🔍 Found 3 relevant chunks.

--- Result 1 (Page 1.0) ---
Table of Contents
1. Scope. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  4
2. Terminology and Conventions . . . . ...

--- Result 2 (Page 1.0) ---
3.10. Parent idTag . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  19
3.11. Reservations . . . . . . . . . . . . . . . . . . . . . . . . . . ...

--- Result 3 (Page 3.0) ---
7.36. Reason. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  90
7.37. RecurrencyKindType . . . . . . . . . . . . . . . . . . . . . ....
